# 06 — Test: crawl 100 source versions

Runs the real pipeline (`05_run_pipeline.ipynb`) on the **first assigned IDs, until they hold
`TEST_VERSIONS` = 100 source versions**, then checks the result. Everything is written to
`test_runs/100_versions/` (on Drive when run on Colab), so the folders of the real crawl are not touched.

What it measures and checks:
- every version was downloaded (`ok` or PDF-only `empty_source`), none failed;
- only `.tex` and `.bib` files are stored in the `vN/` folders;
- `logs/papers.csv` has one row per paper; a second run sends no request (resume);
- download time per version against the 3 s arXiv floor, discovery time and sizes.

**Time:** about 3 s × 100 ≈ 5 minutes (plus a few seconds of discovery). References are not part of
this test: test them with `07_run_references.ipynb` and `REFERENCE_LIMIT`. For a bigger test, change
`TEST_VERSIONS` in the first code cell (e.g. `1000`, about 50 minutes). If Colab disconnects,
run all cells again: the test resumes where it stopped.

## 1. Run the pipeline in test mode

On Colab this cell clones branch `VA` first (same as `05_run_pipeline.ipynb`). It then runs `05` with
`TEST_VERSIONS = 100` and `RUN_REFERENCES = False`: discovery of the first IDs, download, and the
statistics table.

In [ ]:
# On Colab: clone or update branch VA, install packages and work from src/ (the notebooks use relative paths).
import os
import subprocess
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    ON_COLAB = True
except ImportError:
    ON_COLAB = False

if ON_COLAB and not Path("00_config.ipynb").is_file():
    REPO = Path("/content/intro-ds-arxiv-project")
    BRANCH = "VA"
    if not (REPO / ".git").is_dir():
        subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch",
                        "https://github.com/thinhelzan842/intro-ds-arxiv-project.git", str(REPO)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO / "src/requirements.txt")], check=True)
    os.chdir(REPO / "src")

TEST_VERSIONS = 100   # e.g. 1000 for a bigger test (about 50 minutes)
# References are tested on their own with 07_run_references.ipynb (REFERENCE_LIMIT), so this test
# does not wait for Semantic Scholar.
PIPELINE_OVERRIDES = {"TEST_VERSIONS": TEST_VERSIONS, "RUN_REFERENCES": False}
get_ipython().run_line_magic("run", "-i ./05_run_pipeline.ipynb")

## 2. Checks

Fails with a clear message if a version is missing (see `errors.jsonl`, then run again to retry),
if any file other than `.tex`/`.bib` was stored, or if a second run would send requests again.

In [ ]:
import json

assert download_summary is not None, "the download did not run (RUN_DOWNLOAD is False?)"
assert not download_summary["interrupted"], "the run was interrupted: run all cells again to resume"

n_papers = len(papers)
n_versions = sum(p["versions"] for p in papers)
progress = download_summary["progress"]
assert n_versions >= TEST_VERSIONS, f"only {n_versions} versions found in the assigned ranges"
assert progress["versions_total"] == n_versions
assert progress["versions_completed"] == n_versions, (
    f"{n_versions - progress['versions_completed']} versions failed, see {DOWNLOAD_STATE / 'errors.jsonl'}; "
    "run all cells again to retry them")

# Only .tex/.bib inside <paper>/vN/; metadata.json and references.json sit next to the vN/ folders.
stored = [p for p in DATA_DIR.rglob("*") if p.is_file() and len(p.relative_to(DATA_DIR).parts) > 2]
unexpected = [p for p in stored if p.suffix.lower() not in KEEP]
assert not unexpected, f"files other than .tex/.bib were stored: {unexpected[:5]}"
version_dirs = [p for p in DATA_DIR.glob("*/v*") if p.is_dir()]
assert len(version_dirs) == n_versions, (len(version_dirs), n_versions)

with PAPER_LOG.open(newline="", encoding="utf-8") as stream:
    log_rows = list(csv.DictReader(stream))
assert len(log_rows) == n_papers, (len(log_rows), n_papers)

again = download_sources(DOWNLOAD_MANIFEST, DATA_DIR, DOWNLOAD_STATE, user_agent=DOWNLOAD_USER_AGENT)
assert again["versions_skipped"] == n_versions and again["versions_failed_this_run"] == 0
print(f"\nResume check: {again['versions_skipped']:,} versions skipped in {again['elapsed_seconds']:.2f} s, no request sent")
print("\nALL CHECKS PASSED")

## 3. Numbers

Download time counts only the sessions that downloaded something; the estimate scales the measured
time per version to the 20,000 assigned IDs.

In [ ]:
# Timing over every session of this test (the run can be resumed after a disconnect).
history = [json.loads(line) for line in (DOWNLOAD_STATE / "run_history.jsonl").read_text(encoding="utf-8").splitlines()]
download_seconds = sum(h["elapsed_seconds"] for h in history if h["versions_skipped"] < h["progress"]["versions_total"])
receipts = load_receipts(validate_manifest(papers), DATA_DIR, DOWNLOAD_STATE)
before = sum(r["before_bytes"] for r in receipts.values())
after = sum(r["after_bytes"] for r in receipts.values())

rows = [
    ("Papers (IDs in order from the first range)", f"{n_papers:,}"),
    ("Source versions", f"{n_versions:,}"),
    ("Versions with LaTeX / PDF-only", f"{sum(r['status'] == 'ok' for r in receipts.values()):,} / "
                                         f"{sum(r['status'] == 'empty_source' for r in receipts.values()):,}"),
    ("Discovery time (all sessions)", f"{discovery_seconds:,.1f} s"),
    ("Download time (all sessions)", f"{download_seconds / 60:,.1f} min"),
    ("Download time per version (floor: 3 s)", f"{download_seconds / n_versions:.2f} s"),
    ("Download time per paper", f"{download_seconds / n_papers:.2f} s"),
    ("Source size before -> after (.tex/.bib only)", f"{before / 1e6:,.1f} MB -> {after / 1e6:,.1f} MB "
                                                      f"({100 * (1 - after / before):.1f}% saved)" if before else "-"),
    ("Peak RAM (this session)", f"{performance['peak_ram_mb']:.0f} MB"),
    ("Estimate for 20,000 IDs (same versions/paper)", f"{20000 * n_versions / n_papers * download_seconds / n_versions / 3600:.1f} h"),
]
width = max(len(name) for name, _ in rows)
for name, value in rows:
    print(f"{name:<{width}}  {value}")